# Voto mayoritario de Boyer-Moore

Encuentra el elemento que aparece más de $n/2$ veces en una lista en $O(n)$ y $O(1)$ memoria. Mantiene un candidato y un contador: si el contador llega a 0 cambia de candidato; los votos de los demás valores se anulan con los del mayoritario.

## 1. Elemento mayoritario

Devolver el elemento mayoritario o `None` si no existe (hay que verificar el candidato con una segunda pasada).

In [1]:
def mayoritario(lista):
    candidato, votos = None, 0
    for x in lista:
        if votos == 0:
            candidato = x
        votos += 1 if x == candidato else -1
    return candidato if lista.count(candidato) > len(lista) // 2 else None

print(mayoritario([2, 2, 1, 1, 1, 2, 2]), mayoritario([1, 2, 3, 2]))

2 None


## 2. Elementos que superan n/3

Extender la idea con dos candidatos para encontrar todos los elementos que aparecen más de $n/3$ veces (como mucho hay dos).

In [2]:
def mayoritarios_tercio(lista):
    c1, c2, v1, v2 = None, None, 0, 0
    for x in lista:
        if x == c1:
            v1 += 1
        elif x == c2:
            v2 += 1
        elif v1 == 0:
            c1, v1 = x, 1
        elif v2 == 0:
            c2, v2 = x, 1
        else:
            v1, v2 = v1 - 1, v2 - 1
    return [c for c in {c1, c2} if c is not None and lista.count(c) > len(lista) // 3]

print(mayoritarios_tercio([3, 2, 3]), sorted(mayoritarios_tercio([1, 1, 1, 3, 3, 2, 2, 2])))

[3] [1, 2]


## 3. Traza del recuento

Mostrar paso a paso cómo evolucionan el candidato y los votos en `[2, 2, 1, 1, 1, 2, 2]`.

In [3]:
def traza_mayoritario(lista):
    candidato, votos = None, 0
    print(" x  candidato  votos")
    for x in lista:
        if votos == 0:
            candidato = x
        votos += 1 if x == candidato else -1
        print(f"{x:2}  {candidato:9}  {votos:5}")

traza_mayoritario([2, 2, 1, 1, 1, 2, 2])

 x  candidato  votos
 2          2      1
 2          2      2
 1          2      1
 1          2      0
 1          1      1
 2          1      0
 2          2      1


## 4. Comprobación contra fuerza bruta

Comprobar `mayoritario` y `mayoritarios_tercio` con listas aleatorias, contando las apariciones con `Counter`.

In [4]:
import random
from collections import Counter

def mayoritarios_bruto(lista, k):
    return sorted(x for x, c in Counter(lista).items() if c > len(lista) // k)

random.seed(16)
for _ in range(2000):
    datos = random.choices(range(4), k=random.randint(1, 15))
    esperado = mayoritarios_bruto(datos, 2)
    assert mayoritario(datos) == (esperado[0] if esperado else None)
    assert sorted(mayoritarios_tercio(datos)) == mayoritarios_bruto(datos, 3)
print("Comprobadas 2000 listas")

Comprobadas 2000 listas


## 5. Elementos que superan n/k

Generalizar a $k - 1$ candidatos (algoritmo de Misra-Gries) para encontrar los elementos que aparecen más de $n/k$ veces en $O(nk)$ y $O(k)$ memoria. Cuando llega un valor que no es candidato y no hay hueco, se resta un voto a todos: ese valor y $k - 1$ candidatos se anulan entre sí.

In [5]:
def mayoritarios_k(lista, k):
    candidatos = {}
    for x in lista:
        if x in candidatos:
            candidatos[x] += 1
        elif len(candidatos) < k - 1:
            candidatos[x] = 1
        else:
            for c in list(candidatos):
                candidatos[c] -= 1
                if candidatos[c] == 0:
                    del candidatos[c]
    return sorted(c for c in candidatos if lista.count(c) > len(lista) // k)

for _ in range(2000):
    datos = random.choices(range(6), k=random.randint(1, 20))
    k = random.randint(2, 5)
    assert mayoritarios_k(datos, k) == mayoritarios_bruto(datos, k)
print(mayoritarios_k([1, 2, 1, 3, 1, 2, 4, 2, 5, 1], 4))

[1, 2]
